# Enterprise Loan Approval Demo

This demo illustrates how a hybrid AI approach can be applied to loan approval workflows on IBM LinuxONE.

It includes the following:
- Predictive AI (XGBoost GBTree)
- Document processing and feature extraction using generative AI
- Retrieval-augmented generation (RAG)
- Generative AI recommendation and answers

## Architecture Overview

At a high level, this demo does the following:
- Extract document features from an applicant's documents with the help of a generative AI model
- Use predictive AI to make a high/low risk prediction based on structured data.
- Produce a recommendation for approving or rejecting a loan application using the document features and prediction.
- Allow users to ask about the recommendation.

## Environment Setup

Refer to the code in `loan_demo` to see the specifics.

In [ ]:
import pandas as pd

from loan_demo.model_utils import predict_risk
from loan_demo.pdf_utils import extract_text_from_pdf
from loan_demo.rag_utils import (
    load_model,
    load_policy_docs,
    compress_applicant_documents,
    extract_document_features,
    generate_recommendation,
    generate_answer,
    build_policy_index,
    summarize_applicant,
)

In [ ]:
# Load IBM Granite model used for generative AI
model = load_model()

## Load Applicant Documents

Select applicant files and prepare content for feature extraction.
Refer to `sample_data` to choose an applicant to process.

In [ ]:
APPLICANT_ID = 5

loan_df = pd.read_csv("sample_data/sample_loan.csv")
loan = loan_df.iloc[APPLICANT_ID - 1].to_dict()

bank_pdf = (
    f"sample_data/bank_statements/"
    f"{APPLICANT_ID}.pdf"
)

paystub_pdf = (
    f"sample_data/pay_stubs/"
    f"{APPLICANT_ID}.pdf"
)

w2_pdf = (
    f"sample_data/w2/"
    f"{APPLICANT_ID}.pdf"
)

bank_text = extract_text_from_pdf(bank_pdf)
paystub_text = extract_text_from_pdf(paystub_pdf)
w2_text = extract_text_from_pdf(w2_pdf)

# additional preprocessing and cleanup of extracted text
document_text = compress_applicant_documents(
    bank_text,
    paystub_text,
    w2_text,
)

print(document_text)

## Extract Financial Features

The LLM analyzes applicant information and produces a set of financial features.

In [ ]:
# Use the model to extract features from unstructured documents
document_features = (
    extract_document_features(
        model,
        document_text,
    )
)

## Predict Risk

An XGBoost GBTree model generates a traditional risk score using applicant features

In [ ]:
risk_result = predict_risk(loan)
print(risk_result)

## Retrieve Relevant Lending Policies

Using BM25 retrieval, we identify supporting policy guidance for the recommendation step. This strategy allows a model like IBM Granite to have access to a huge volume of information while only retrieving the information most relevant.

In [ ]:
policies = load_policy_docs("sample_data/policy")

bm25, policy_chunks = build_policy_index(
    policies,
    chunk_size=300,
)

## Recommendation

Now we combine the analysis of the applicant's financial documents, the risk model prediction, and the retrieved policy guidance to produce a final recommendation in natural language.

In [ ]:
recommendation = generate_recommendation(
    model=model,
    applicant=loan,
    risk_result=risk_result,
    document_features=document_features,
    bm25=bm25,
    policy_chunks=policy_chunks,
)

## Ask Questions About the Application

Natural-language question answering over the generated decision and context.

In [ ]:
# Preprocessing step as many fields are not relevant
applicant_summary = summarize_applicant(loan)

# Build context for model
context = f"""
Applicant:
{applicant_summary}

Risk:
{risk_result}

Document Analysis:
{document_features}

Recommendation:
{recommendation}
"""

# Change this question to ask different questions
response = generate_answer(
    model,
    context,
    "Why was this applicant considered low risk?"
)